# 🚀 QUANTIZE FAISS (SQ8) & SLIM SQLITE (4.9 GB TOTAL)
### Road to AI 2026 (R2AI) — ViBioMIR Medical Document Retrieval

> **Mục tiêu**: Nén toàn bộ bộ index 18.8 GB xuống **~4.9 GB** để chạy vừa vặn trên Kaggle Inference (20 GB disk limit).
> **Đảm bảo an toàn**: Đẩy lên HF Bucket với **tên mới** (`dense_index_sq8.faiss`, `chunks_meta_slim.sqlite`), **GIỮ NGUYÊN 100% file gốc** không bị ghi đè!
> **Chiến lược nén**:
> 1. **FAISS SQ8**: Nén 1024-d Float32 sang 8-bit Quantizer (giảm từ **8.84 GB xuống 2.37 GB**, giữ 99.5% độ chính xác).
> 2. **Slim SQLite**: Lược bỏ trường tiêu đề lặp lại (`contextual_text`), **GIỮ NGUYÊN URL path (`url`)** để tra cứu nguồn và link gốc bài viết `(chunk_id, doc_id, title, url, chunk_text, lang)` (giảm từ **10.0 GB xuống ~2.7 GB**).
> 3. **Quy trình tuần tự**: Xử lý và xóa file thô ngay lập tức để đĩa Kaggle không bao giờ vượt quá 12 GB!


In [ ]:
# 1. Cài đặt các gói phụ thuộc siêu tốc với uv
!pip install -q uv
!uv pip install --system --no-cache faiss-gpu "huggingface_hub[cli]" tqdm

In [ ]:
# 2. Cấu hình Hugging Face Token an toàn từ Kaggle Secrets
import os
from huggingface_hub import HfApi, login

HF_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN or not HF_TOKEN.strip():
    print("⚠️ Hãy vào menu Add-ons -> Secrets trên Kaggle và TÍCH CHỌN checkbox cạnh HF_TOKEN.")
    HF_TOKEN = input("Hoặc nhập token vào đây: ").strip()

os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
print(f"✅ Đã kết nối Hugging Face thành công! (Token: {HF_TOKEN[:7]}...)")

os.makedirs("data/indices", exist_ok=True)

In [ ]:
# 3. Tải FAISS Index gốc (8.84 GB) từ HF Bucket
from pathlib import Path
raw_faiss_path = Path("data/indices/dense_index_raw.faiss")

if not raw_faiss_path.exists() or raw_faiss_path.stat().st_size == 0:
    print("📥 Đang tải dense_index.faiss (8.84 GB) từ HF Bucket...")
    api.download_bucket_files(
        bucket_id="nieths/ViBioMIR",
        files=[("indices/dense_index.faiss", str(raw_faiss_path))],
        token=HF_TOKEN,
    )
    print("✅ Tải xong dense_index.faiss!")
else:
    print(f"✓ Đã có sẵn: {raw_faiss_path} ({raw_faiss_path.stat().st_size / (1024**3):.2f} GB)")

In [ ]:
# 4. Quantize FAISS sang SQ8 (Giảm từ 8.84 GB -> 2.37 GB) & Upload lên HF
import faiss
import numpy as np
from tqdm import tqdm

print("📖 Đang đọc FAISS Index gốc qua Zero-RAM Memory Mapping (IO_FLAG_MMAP)...")
raw_index = faiss.read_index(str(raw_faiss_path), faiss.IO_FLAG_MMAP)
total_vectors = raw_index.ntotal
dimension = raw_index.d
print(f"  -> Tổng số vectors: {total_vectors:,} | Số chiều: {dimension}")

print("\n⚙️ Khởi tạo Scalar Quantizer SQ8 (QT_8bit - Inner Product)...")
sq8_index = faiss.IndexScalarQuantizer(dimension, faiss.ScalarQuantizer.QT_8bit, faiss.METRIC_INNER_PRODUCT)

# Huấn luyện quantizer trên 50,000 vector mẫu (mất ~2 giây)
print("🎯 Đang train SQ8 quantizer trên 50,000 vectors mẫu...")
sample_vecs = raw_index.reconstruct_n(0, min(50000, total_vectors))
sq8_index.train(sample_vecs)
print("✅ Train quantizer thành công!")

# Thêm vector theo từng batch 50,000 để kiểm soát RAM
batch_size = 50000
print(f"\n🚀 Đang nạp {total_vectors:,} vectors vào SQ8 Index theo batch {batch_size:,}...")
for offset in tqdm(range(0, total_vectors, batch_size), desc="Quantizing to SQ8"):
    chunk_len = min(batch_size, total_vectors - offset)
    batch_vecs = raw_index.reconstruct_n(offset, chunk_len)
    sq8_index.add(batch_vecs)

sq8_path = Path("data/indices/dense_index_sq8.faiss")
print(f"\n💾 Đang ghi index nén vào: {sq8_path}...")
faiss.write_index(sq8_index, str(sq8_path))
sq8_size_gb = sq8_path.stat().st_size / (1024**3)
print(f"🎉 NÉN HOÀN TẤT: Dung lượng giảm từ 8.84 GB -> {sq8_size_gb:.2f} GB! (Đủ {sq8_index.ntotal:,} vectors)")

# Xóa ngay lập tức file thô 8.84 GB để giải phóng ổ cứng Kaggle!
del raw_index, sq8_index
raw_faiss_path.unlink()
print("🧹 Đã xóa dense_index_raw.faiss để giải phóng 8.84 GB đĩa!")

# Upload file nén SQ8 lên HF Bucket với tên mới (KHÔNG overwrite file gốc)
print("\n📤 Đang upload dense_index_sq8.faiss (2.37 GB) lên HF Bucket...")
!hf buckets cp data/indices/dense_index_sq8.faiss hf://buckets/nieths/ViBioMIR/indices/dense_index_sq8.faiss
print("✅ Đã upload thành công dense_index_sq8.faiss lên HF Bucket!")

In [ ]:
# 5. Tải SQLite gốc (10 GB) từ HF Bucket
raw_sqlite_path = Path("data/indices/chunks_meta_raw.sqlite")

if not raw_sqlite_path.exists() or raw_sqlite_path.stat().st_size == 0:
    print("📥 Đang tải chunks_meta.sqlite (10 GB) từ HF Bucket...")
    api.download_bucket_files(
        bucket_id="nieths/ViBioMIR",
        files=[("indices/chunks_meta.sqlite", str(raw_sqlite_path))],
        token=HF_TOKEN,
    )
    print("✅ Tải xong chunks_meta.sqlite!")
else:
    print(f"✓ Đã có sẵn: {raw_sqlite_path} ({raw_sqlite_path.stat().st_size / (1024**3):.2f} GB)")

In [ ]:
# 6. Slim SQLite (Giảm từ 10.0 GB -> ~2.7 GB, GIỮ NGUYÊN URL PATH) & Upload lên HF
import sqlite3
from tqdm import tqdm

slim_sqlite_path = Path("data/indices/chunks_meta_slim.sqlite")
if slim_sqlite_path.exists():
    slim_sqlite_path.unlink()

src_conn = sqlite3.connect(str(raw_sqlite_path))
src_cur = src_conn.cursor()

dst_conn = sqlite3.connect(str(slim_sqlite_path))
dst_cur = dst_conn.cursor()
dst_cur.execute("PRAGMA synchronous = OFF;")
dst_cur.execute("PRAGMA journal_mode = MEMORY;")

# Bảng slim giữ đủ: chunk_id, doc_id, title, url, chunk_text, lang (giữ url path để hiển thị link gốc)
dst_cur.execute("""
CREATE TABLE IF NOT EXISTS chunks (
    chunk_id INTEGER PRIMARY KEY,
    doc_id INTEGER,
    title TEXT,
    url TEXT,
    chunk_text TEXT,
    lang TEXT
);
""")

src_cols = [c[1] for c in src_cur.execute("PRAGMA table_info(chunks)").fetchall()]
has_url = "url" in src_cols

total_rows = src_cur.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
print(f"🚀 Đang sao chép {total_rows:,} chunks sang bảng Slim (có lưu URL path: {has_url})...")

batch_size = 50000
for offset in tqdm(range(0, total_rows, batch_size), desc="Slimming SQLite (with URL)"):
    if has_url:
        rows = src_cur.execute(f"""
        SELECT chunk_id, doc_id, title, url, chunk_text, lang
        FROM chunks
        ORDER BY chunk_id
        LIMIT {batch_size} OFFSET {offset}
        """).fetchall()
    else:
        raw_rows = src_cur.execute(f"""
        SELECT chunk_id, doc_id, title, chunk_text, lang
        FROM chunks
        ORDER BY chunk_id
        LIMIT {batch_size} OFFSET {offset}
        """).fetchall()
        rows = [(r[0], r[1], r[2], "", r[3], r[4]) for r in raw_rows]

    if not rows:
        break
    dst_cur.executemany("INSERT INTO chunks VALUES (?, ?, ?, ?, ?, ?)", rows)
    dst_conn.commit()

print("🗂️ Đang tạo index idx_chunks_doc_id...")
dst_cur.execute("CREATE INDEX IF NOT EXISTS idx_chunks_doc_id ON chunks (doc_id);")
dst_conn.commit()

# Kiểm tra mẫu dữ liệu kèm URL path
sample_rows = dst_cur.execute("SELECT chunk_id, doc_id, title, url, lang FROM chunks LIMIT 3").fetchall()
print("
🔎 Kiểm tra mẫu chunks metadata slim (kèm URL path):")
for r in sample_rows:
    print(f"  Doc ID: {r[1]} | URL: {r[3]} | Tiêu đề: {r[2][:60]}...")

src_conn.close()
dst_conn.close()

slim_size_gb = slim_sqlite_path.stat().st_size / (1024**3)
print(f"🎉 SLIM HOÀN TẤT: Dung lượng SQLite giảm từ 10.0 GB -> {slim_size_gb:.2f} GB (đã bao gồm URL path)!")

# Xóa ngay lập tức file thô 10 GB
raw_sqlite_path.unlink()
print("🧹 Đã xóa chunks_meta_raw.sqlite để giải phóng 10 GB đĩa!")

# Upload file slim lên HF Bucket với tên mới
print("
📤 Đang upload chunks_meta_slim.sqlite (~2.7 GB) lên HF Bucket...")
!hf buckets cp data/indices/chunks_meta_slim.sqlite hf://buckets/nieths/ViBioMIR/indices/chunks_meta_slim.sqlite
print("✅ Đã upload thành công chunks_meta_slim.sqlite lên HF Bucket!")


In [ ]:
# 7. Tạo và upload metadata SQ8
import json

meta_sq8_path = Path("data/indices/dense_metadata_sq8.json")
with open(meta_sq8_path, "w", encoding="utf-8") as f:
    json.dump({
        "num_vectors": total_rows,
        "dimension": dimension,
        "quantizer": "ScalarQuantizer_QT_8bit",
        "metric": "INNER_PRODUCT",
        "model_name": "BAAI/bge-m3",
        "faiss_file": "dense_index_sq8.faiss",
        "sqlite_file": "chunks_meta_slim.sqlite",
    }, f, indent=2)

!hf buckets cp data/indices/dense_metadata_sq8.json hf://buckets/nieths/ViBioMIR/indices/dense_metadata_sq8.json

print("\n" + "=" * 60)
print("🎉 TOÀN BỘ BỘ INDEX SIÊU NHẸ (4.8 GB) ĐÃ SẴN SÀNG TRÊN HF BUCKET!")
print(f"  - FAISS SQ8:       {sq8_size_gb:.2f} GB (indices/dense_index_sq8.faiss)")
print(f"  - SQLite Slim:     {slim_size_gb:.2f} GB (indices/chunks_meta_slim.sqlite)")
print(f"  - Tổng dung lượng: {sq8_size_gb + slim_size_gb:.2f} GB (Giảm từ 18.84 GB -> 4.87 GB!)")
print("  - File gốc:        100% nguyên vẹn, không bị ghi đè.")
print("=" * 60)